# Credit Risk Modelling — Part 4.9: Save Artefacts & Inference Script
---
**Objective:** Package everything needed to deploy and use the best model in production.

This notebook produces three things:
1. **Clean saved artefacts** — model pickle, preprocessing metadata, threshold
2. **`inference.py`** — a standalone script that loads the artefacts and scores any new applicant dataset
3. **Validation** — a dry run proving the inference script produces identical results to the notebook

After this notebook, anyone with `inference.py` + the two pickle files can score new data  
without needing any of the training notebooks.

**Steps:**
1. Load best model and all artefacts
2. Save final model artefacts cleanly
3. Write `inference.py`
4. Dry-run validation — score test set via inference script, compare to notebook predictions
5. Document the inference API
6. Deployment checklist

## 1 — Imports & Load Everything

In [1]:
import numpy as np
import pandas as pd
import joblib, json, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Load all artefacts
X_test         = np.load("X_test.npy")
y_test         = np.load("y_test.npy")
meta           = joblib.load("preprocessing_meta.pkl")
results        = joblib.load("results_so_far.pkl")
with open("model_selection.json") as f:
    selection = json.load(f)

best_name   = selection["best_model_name"]
best_result = results[best_name]
best_model  = best_result["model"]
opt_thresh  = best_result["opt_thresh"]
feature_cols = meta["feature_cols_final"]

print(f"Best model     : {best_name}")
print(f"Test AUC       : {best_result['test_auc']:.4f}")
print(f"Opt threshold  : {opt_thresh:.4f}")
print(f"Feature count  : {len(feature_cols)}")
print("✓ All artefacts loaded")

Best model     : Logistic Regression
Test AUC       : 0.8575
Opt threshold  : 0.5463
Feature count  : 198
✓ All artefacts loaded


---
## 2 — Save Final Model Artefacts

We save two files — everything inference needs:
- `best_model.pkl` — the fitted model object
- `inference_config.pkl` — preprocessing params + threshold + metadata

In [2]:
# ── Save best model ──────────────────────────────────────────────
joblib.dump(best_model, "best_model.pkl")
print("✓ Saved: best_model.pkl")

# ── Build inference config (all params inference.py needs) ────────
inference_config = {
    # Feature schema
    "num_cols"            : meta["num_cols"],
    "cat_cols"            : meta["cat_cols"],
    "flag_cols"           : meta["flag_cols"],
    "feature_cols_final"  : meta["feature_cols_final"],

    # Fitted preprocessing params (train-derived)
    "medians"             : meta["medians"],      # dict: col -> median value
    "modes"               : meta["modes"],         # dict: col -> fill value

    # Decision threshold
    "opt_thresh"          : opt_thresh,

    # Model metadata
    "model_name"          : best_name,
    "test_auc"            : best_result["test_auc"],
    "test_ap"             : best_result["test_ap"],
    "ks"                  : best_result["ks"],
    "gini"                : best_result["gini"],
    "cv_mean"             : best_result["cv_mean"],
    "n_features"          : len(feature_cols),
    "missing_threshold"   : meta["missing_threshold"],
}

joblib.dump(inference_config, "inference_config.pkl")
print("✓ Saved: inference_config.pkl")

# Confirm file sizes
for fname in ["best_model.pkl", "inference_config.pkl"]:
    size_kb = Path(fname).stat().st_size / 1024
    print(f"  {fname:<30}: {size_kb:>8.1f} KB")

✓ Saved: best_model.pkl
✓ Saved: inference_config.pkl
  best_model.pkl                :      6.7 KB
  inference_config.pkl          :      6.3 KB


---
## 3 — Write `inference.py`

This is the production scoring script. It:
- Accepts a raw CSV of new applicants (same schema as `X_train`)
- Loads `best_model.pkl` and `inference_config.pkl`
- Runs the full preprocessing pipeline
- Outputs a CSV with predicted probability + binary flag at optimal threshold
- Validates the input schema before scoring

In [5]:
# Write inference.py to disk
inference_script_lines = [
    "#!/usr/bin/env python3",
    '"""',
    "inference.py -- Credit Risk Model Scoring Script",
    "================================================",
    "Scores new applicant data using the trained credit risk model.",
    "",
    "Usage:",
    "    python inference.py --input new_applicants.csv --output scores.csv",
    "",
    "Required files (must be in same directory):",
    "    best_model.pkl        -- trained model",
    "    inference_config.pkl  -- preprocessing params and threshold",
    "",
    "Input CSV format:",
    "    Same column schema as X_train (account_id + all feature columns).",
    "    Missing values (NaN / blank) are handled automatically.",
    "",
    "Output CSV columns:",
    "    account_id            -- original identifier",
    "    default_probability   -- predicted probability of default (0-1)",
    "    high_risk_flag        -- 1 if probability >= opt_thresh, else 0",
    "    risk_band             -- LOW / MEDIUM / HIGH / VERY HIGH",
    '"""',
    "",
    "import argparse",
    "import sys",
    "import numpy as np",
    "import pandas as pd",
    "import joblib",
    "import warnings",
    "from pathlib import Path",
    "",
    "warnings.filterwarnings('ignore')",
    "",
    "RISK_BANDS = [",
    "    (0.00, 0.05, 'LOW'),",
    "    (0.05, 0.15, 'MEDIUM'),",
    "    (0.15, 0.30, 'HIGH'),",
    "    (0.30, 1.01, 'VERY HIGH'),",
    "]",
    "",
    "def assign_risk_band(prob):",
    "    for lo, hi, band in RISK_BANDS:",
    "        if lo <= prob < hi:",
    "            return band",
    "    return 'VERY HIGH'",
    "",
    "def load_artefacts(model_path='best_model.pkl', config_path='inference_config.pkl'):",
    "    if not Path(model_path).exists():",
    "        raise FileNotFoundError(f'Model file not found: {model_path}')",
    "    if not Path(config_path).exists():",
    "        raise FileNotFoundError(f'Config file not found: {config_path}')",
    "    model  = joblib.load(model_path)",
    "    config = joblib.load(config_path)",
    "    print(f\"[INFO] Loaded model  : {config['model_name']}\")",
    "    print(f\"[INFO] Model AUC     : {config['test_auc']:.4f}\")",
    "    print(f\"[INFO] Threshold     : {config['opt_thresh']:.4f}\")",
    "    return model, config",
    "",
    "def validate_schema(df, config):",
    "    required = set(config['num_cols'] + config['cat_cols'] + ['account_id'])",
    "    present  = set(df.columns)",
    "    missing  = required - present",
    "    extra    = present - required - {'account_id'}",
    "    if missing:",
    "        raise ValueError(f'Input CSV missing {len(missing)} required columns: {sorted(missing)[:10]}')",
    "    if extra:",
    "        print(f'[WARN] {len(extra)} extra columns will be ignored')",
    "    print(f'[INFO] Schema OK -- {len(df.columns)} columns, {len(df):,} rows')",
    "",
    "def preprocess(df, config):",
    "    df        = df.copy()",
    "    num_cols  = config['num_cols']",
    "    cat_cols  = config['cat_cols']",
    "    flag_cols = config['flag_cols']",
    "    medians   = pd.Series(config['medians'])",
    "    modes     = config['modes']",
    "    feat_cols = config['feature_cols_final']",
    "    for col in flag_cols:",
    "        if col in df.columns:",
    "            df[f'{col}_is_missing'] = df[col].isnull().astype('int8')",
    "    df[num_cols] = df[num_cols].fillna(medians)",
    "    if cat_cols:",
    "        for c in cat_cols:",
    "            df[c] = df[c].fillna(modes.get(c, 'MISSING'))",
    "        df = pd.get_dummies(df, columns=cat_cols, drop_first=False)",
    "    for c in feat_cols:",
    "        if c not in df.columns:",
    "            df[c] = 0",
    "    X = df[feat_cols].values.astype('float32')",
    "    nan_count = np.isnan(X).sum()",
    "    if nan_count > 0:",
    "        print(f'[WARN] {nan_count} NaN remain after imputation -- replacing with 0')",
    "        X = np.nan_to_num(X, nan=0.0)",
    "    return X",
    "",
    "def score(input_path, output_path, model_path='best_model.pkl', config_path='inference_config.pkl'):",
    "    print(f'[INFO] Input  : {input_path}')",
    "    print(f'[INFO] Output : {output_path}')",
    "    print()",
    "    model, config = load_artefacts(model_path, config_path)",
    "    df = pd.read_csv(input_path)",
    "    print(f'[INFO] Loaded {len(df):,} rows, {len(df.columns)} columns')",
    "    validate_schema(df, config)",
    "    account_ids = df['account_id'].values",
    "    X      = preprocess(df, config)",
    "    probas = model.predict_proba(X)[:, 1]",
    "    flags  = (probas >= config['opt_thresh']).astype(int)",
    "    bands  = [assign_risk_band(p) for p in probas]",
    "    output_df = pd.DataFrame({",
    "        'account_id'        : account_ids,",
    "        'default_probability': probas.round(6),",
    "        'high_risk_flag'    : flags,",
    "        'risk_band'         : bands,",
    "    })",
    "    output_df.to_csv(output_path, index=False)",
    "    print()",
    "    print('[INFO] Scoring complete')",
    "    print(f'  Total scored      : {len(output_df):,}')",
    "    print(f'  High risk flagged : {flags.sum():,} ({flags.mean()*100:.1f}%)')",
    "    for band in ['LOW','MEDIUM','HIGH','VERY HIGH']:",
    "        n = (output_df['risk_band']==band).sum()",
    "        print(f'    {band:<12}: {n:>6,}  ({n/len(output_df)*100:.1f}%)')",
    "    print(f'  Output written to : {output_path}')",
    "    return output_df",
    "",
    "if __name__ == '__main__':",
    "    parser = argparse.ArgumentParser(description='Credit risk model scoring')",
    "    parser.add_argument('--input',  required=True,  help='Path to input CSV')",
    "    parser.add_argument('--output', required=True,  help='Path to output CSV')",
    "    parser.add_argument('--model',  default='best_model.pkl')",
    "    parser.add_argument('--config', default='inference_config.pkl')",
    "    args = parser.parse_args()",
    "    score(args.input, args.output, args.model, args.config)",
]

inference_script = "\n".join(inference_script_lines)

with open("inference.py", "w") as f:
    f.write(inference_script)

print("Saved: inference.py")
print(f"  Lines : {len(inference_script_lines)}")
print()
print("Usage:")
print("  python inference.py --input new_applicants.csv --output scores.csv")


Saved: inference.py
  Lines : 134

Usage:
  python inference.py --input new_applicants.csv --output scores.csv


---
## 4 — Dry-Run Validation

We run the inference script on the test set and compare its output  
to the predictions made in the notebook.  
If they match, the script is correct.

In [6]:
# Import the score function directly from inference.py
import importlib.util, sys

spec   = importlib.util.spec_from_file_location("inference", "inference.py")
inf_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(inf_mod)

# Re-load raw test data (inference.py starts from raw CSV)


TEST_X = "/Users/parveenkumarsharma/Documents/Ecom_project/Credit_risk_model/MP Credit Modelling Takehome Task (train + test) - X_test.csv"

TEST_Y = "/Users/parveenkumarsharma/Documents/Ecom_project/Credit_risk_model/MP Credit Modelling Takehome Task (train + test) - Y_test.csv"


print("Running inference.py on test set...")
print("-" * 45)
output_df = inf_mod.score(TEST_X, "dry_run_scores.csv")
print("-" * 45)
print()

# Compare to notebook predictions
notebook_proba = best_result["proba_test"]
infer_proba    = output_df["default_probability"].values

# Align by account_id (order may differ)
test_df  = pd.read_csv(TEST_X)
y_test_df = pd.read_csv(TEST_Y)
test_merged = test_df.merge(y_test_df, on="account_id")

output_df_indexed = output_df.set_index("account_id")
notebook_df = pd.DataFrame({
    "account_id"     : test_merged["account_id"].values,
    "notebook_proba" : notebook_proba,
}).set_index("account_id")

compare = notebook_df.join(output_df_indexed[["default_probability"]])
compare.columns = ["notebook", "inference"]
compare["diff"] = (compare["notebook"] - compare["inference"]).abs()

print("Validation results:")
print(f"  Rows compared        : {len(compare):,}")
print(f"  Max absolute diff    : {compare['diff'].max():.8f}")
print(f"  Mean absolute diff   : {compare['diff'].mean():.8f}")
print(f"  Identical (< 1e-5)   : {(compare['diff'] < 1e-5).sum():,} / {len(compare):,}")
print()
if compare["diff"].max() < 1e-4:
    print("✓ VALIDATION PASSED -- inference.py produces identical predictions")
else:
    print("✗ VALIDATION FAILED -- predictions differ, check preprocessing pipeline")

Running inference.py on test set...
---------------------------------------------
[INFO] Input  : /Users/parveenkumarsharma/Documents/Ecom_project/Credit_risk_model/MP Credit Modelling Takehome Task (train + test) - X_test.csv
[INFO] Output : dry_run_scores.csv

[INFO] Loaded model  : Logistic Regression
[INFO] Model AUC     : 0.8575
[INFO] Threshold     : 0.5463
[INFO] Loaded 4,000 rows, 101 columns
[INFO] Schema OK -- 101 columns, 4,000 rows

[INFO] Scoring complete
  Total scored      : 4,000
  High risk flagged : 1,096 (27.4%)
    LOW         :  1,167  (29.2%)
    MEDIUM      :    553  (13.8%)
    HIGH        :    515  (12.9%)
    VERY HIGH   :  1,765  (44.1%)
  Output written to : dry_run_scores.csv
---------------------------------------------

Validation results:
  Rows compared        : 4,000
  Max absolute diff    : 0.00000054
  Mean absolute diff   : 0.00000025
  Identical (< 1e-5)   : 4,000 / 4,000

✓ VALIDATION PASSED -- inference.py produces identical predictions


## 5 — Inspect Inference Output

In [7]:
# Load and display the scored output
scores = pd.read_csv("dry_run_scores.csv")

print(f"Output shape: {scores.shape}")
print()
print("First 10 rows:")
print(scores.head(10).to_string(index=False))
print()
print("Risk band distribution:")
band_counts = scores["risk_band"].value_counts().reindex(["LOW","MEDIUM","HIGH","VERY HIGH"])
for band, count in band_counts.items():
    bar = "█" * int(count/len(scores)*40)
    print(f"  {band:<12}: {count:>5,} ({count/len(scores)*100:>5.1f}%)  {bar}")
print()
print("Score statistics:")
print(scores["default_probability"].describe().round(4).to_string())

Output shape: (4000, 4)

First 10 rows:
 account_id  default_probability  high_risk_flag risk_band
    8124511             0.002232               0       LOW
    8089749             0.286656               0      HIGH
    8089843             0.002191               0       LOW
    8117307             0.500138               0 VERY HIGH
    8099551             0.590148               1 VERY HIGH
    8113371             0.717596               1 VERY HIGH
    8107505             0.414179               0 VERY HIGH
    8129065             0.535639               0 VERY HIGH
    8128832             0.005331               0       LOW
    8095442             0.297135               0      HIGH

Risk band distribution:
  LOW         : 1,167 ( 29.2%)  ███████████
  MEDIUM      :   553 ( 13.8%)  █████
  HIGH        :   515 ( 12.9%)  █████
  VERY HIGH   : 1,765 ( 44.1%)  █████████████████

Score statistics:
count    4000.0000
mean        0.3186
std         0.2998
min         0.0000
25%         0.0314
50

---
## 6 — Artefact Manifest & Deployment Checklist

In [9]:
import os

print("=" * 65)
print("  ARTEFACT MANIFEST")
print("=" * 65)

artefacts = {
    "best_model.pkl"       : "Fitted model -- load with joblib.load()",
    "inference_config.pkl" : "Preprocessing params + threshold",
    "inference.py"         : "Standalone scoring script",
    "model_selection.json" : "Selection decision + all metrics",
    "preprocessing_meta.pkl": "Full preprocessing metadata (training)",
}

for fname, desc in artefacts.items():
    exists = Path(fname).exists()
    size   = Path(fname).stat().st_size / 1024 if exists else 0
    icon   = "✓" if exists else "✗  MISSING"
    print(f"  {icon}  {fname:<30} {size:>8.1f} KB   {desc}")

print()
print("=" * 65)
print("  DEPLOYMENT CHECKLIST")
print("=" * 65)
checklist = [
    ("best_model.pkl present",             Path("best_model.pkl").exists()),
    ("inference_config.pkl present",        Path("inference_config.pkl").exists()),
    ("inference.py present",                Path("inference.py").exists()),
    ("Dry-run validation passed",           True),  # confirmed above
    ("Input schema documented",             True),
    ("Output schema documented",            True),
    ("Threshold set and justified",         True),
    ("Model card produced (Part 4.8)",      True),
    ("Retraining cadence defined",          True),
]
for item, done in checklist:
    icon = "✓" if done else "✗"
    print(f"  {icon}  {item}")

print()
print("Minimum files needed to score new data:")
print("  1. best_model.pkl")
print("  2. inference_config.pkl")
print("  3. inference.py")
print()
print("Command:")
print("  python inference.py --input new_data.csv --output scores.csv")

  ARTEFACT MANIFEST
  ✓  best_model.pkl                      6.7 KB   Fitted model -- load with joblib.load()
  ✓  inference_config.pkl                6.3 KB   Preprocessing params + threshold
  ✓  inference.py                        5.0 KB   Standalone scoring script
  ✓  model_selection.json                0.5 KB   Selection decision + all metrics
  ✓  preprocessing_meta.pkl              6.1 KB   Full preprocessing metadata (training)

  DEPLOYMENT CHECKLIST
  ✓  best_model.pkl present
  ✓  inference_config.pkl present
  ✓  inference.py present
  ✓  Dry-run validation passed
  ✓  Input schema documented
  ✓  Output schema documented
  ✓  Threshold set and justified
  ✓  Model card produced (Part 4.8)
  ✓  Retraining cadence defined

Minimum files needed to score new data:
  1. best_model.pkl
  2. inference_config.pkl
  3. inference.py

Command:
  python inference.py --input new_data.csv --output scores.csv


---
## 7 — Inference API Documentation

### Input format
The input CSV must contain `account_id` plus all original feature columns  
(same schema as `X_train`). Missing values are handled automatically.

```
account_id, score_1, balance_1, balance_2, ..., financial_situation_4, financial_situation_5
12345,      720.5,   1500.0,    NaN,       ..., P1,                    B
12346,      NaN,     NaN,       800.0,     ..., S2,                    E
```

### Output format
```
account_id, default_probability, high_risk_flag, risk_band
12345,      0.042314,            0,              LOW
12346,      0.231456,            1,              HIGH
```

| Column | Type | Description |
|---|---|---|
| `account_id` | original | Passed through unchanged |
| `default_probability` | float [0,1] | Predicted probability of default within 12 months |
| `high_risk_flag` | int {0,1} | 1 if `default_probability >= opt_thresh` |
| `risk_band` | string | LOW / MEDIUM / HIGH / VERY HIGH |

### Risk bands
| Band | Probability range | Interpretation |
|---|---|---|
| LOW | 0.00 – 0.05 | Approve with standard terms |
| MEDIUM | 0.05 – 0.15 | Approve with monitoring or reduced limit |
| HIGH | 0.15 – 0.30 | Review manually or decline |
| VERY HIGH | 0.30+ | Decline or require additional verification |

### Python API (without CLI)
```python
from inference import score, load_artefacts, preprocess

# Score a file
output_df = score("new_applicants.csv", "scores.csv")

# Or use components directly
model, config = load_artefacts()
X = preprocess(raw_df, config)
probas = model.predict_proba(X)[:, 1]
```

---
## Summary

### Files produced by this notebook

| File | Purpose | Required for inference |
|---|---|---|
| `best_model.pkl` | Fitted model object | ✓ YES |
| `inference_config.pkl` | Preprocessing params + threshold | ✓ YES |
| `inference.py` | Standalone scoring script | ✓ YES |
| `model_selection.json` | Selection decision record | Documentation |
| `dry_run_scores.csv` | Validation output | Testing only |

### What `inference.py` does
1. Loads `best_model.pkl` and `inference_config.pkl`
2. Validates input CSV schema against training schema
3. Applies the full preprocessing pipeline (missingness flags → median imputation → OHE → column alignment) using **train-fitted parameters only**
4. Scores with `model.predict_proba(X)[:, 1]`
5. Applies optimal threshold and risk band assignment
6. Writes output CSV with `default_probability`, `high_risk_flag`, `risk_band`

### Validation
The dry-run confirmed inference.py produces predictions **identical to the notebook** — maximum absolute difference < 1e-4.

---
**Next → Part 4.10: Final Summary & Reflections**  
Answer the task questions, discuss alternative approaches, and what additional data would help.